In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# creating state for prompt chaining workflow 
class promptState(TypedDict):
    title : str
    outline : str 
    blog : str 
    evaluation : int



In [6]:
# creating a function to generate outline
model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

def generate_outline(state: promptState) -> promptState:
    # fetch
    title = state['title']

    # create prompt 
    prompt = f'generate a short outline on the given topic {title}'

    # generate outline
    answer = model.invoke(prompt).content

    # update state
    state['outline'] = answer

    return state

In [7]:
# function to generate_blog 

def generate_blog(state: promptState) -> promptState:
    # fetch
    title = state['title']
    outline = state['outline']

    # create prompt 
    prompt = f'create a short blog on the topic - {title} using the given outline - {outline}'

    # generate outline
    answer = model.invoke(prompt).content

    # update state
    state['blog'] = answer

    return state

In [8]:
# creating fuction to evaluate (rate our blog)
def evaluation(state: promptState) -> promptState:
    # fetch
    title = state['title']
    outline = state['outline']
    blog = state['blog']

    # create prompt 
    prompt = f'rate my short blog based this  outline - {outline}  , evaluate my blog out of 10 , only give the rating out of 10'

    # generate outline
    answer = model.invoke(prompt).content

    # update state
    state['evaluation'] = answer

    return state

In [10]:
# graph create
graph = StateGraph(promptState)

#add node 
graph.add_node('generate_outline', generate_outline)
graph.add_node('generate_blog', generate_blog)
graph.add_node('evaluation', evaluation)

#add graph 
graph.add_edge(START , 'generate_outline')
graph.add_edge('generate_outline' , 'generate_blog')
graph.add_edge('generate_blog' , 'evaluation')
graph.add_edge('evaluation', END)

# compile
workflow = graph.compile()



In [11]:
initial_state = {'title': 'rising of AI in india'}
final_state = workflow.invoke(initial_state)

In [12]:
final_state

{'title': 'rising of AI in india',
 'outline': 'Here\'s a short outline on the rising of AI in India:\n\n**I. Introduction to AI in India**\n    *   Rapid growth and adoption of Artificial Intelligence across various sectors.\n    *   India\'s vision to become a global hub for AI innovation and implementation ("AI for All").\n    *   Key drivers: large data pool, skilled workforce, government support.\n\n**II. Key Drivers of AI Growth**\n    *   **Government Initiatives:** "National Strategy for AI," NITI Aayog\'s focus, "Digital India" push.\n    *   **Talent Pool:** Abundant STEM graduates, growing AI/ML expertise.\n    *   **Digital Infrastructure:** Expanding internet penetration, smartphone usage, UPI.\n    *   **Startup Ecosystem:** Thriving tech startup scene attracting significant investment.\n    *   **Data Availability:** Vast population generating diverse datasets.\n\n**III. Major Application Areas**\n    *   **Healthcare:** Diagnostics, drug discovery, remote patient monito

In [15]:
print(final_state['outline'])

Here's a short outline on the rising of AI in India:

**I. Introduction to AI in India**
    *   Rapid growth and adoption of Artificial Intelligence across various sectors.
    *   India's vision to become a global hub for AI innovation and implementation ("AI for All").
    *   Key drivers: large data pool, skilled workforce, government support.

**II. Key Drivers of AI Growth**
    *   **Government Initiatives:** "National Strategy for AI," NITI Aayog's focus, "Digital India" push.
    *   **Talent Pool:** Abundant STEM graduates, growing AI/ML expertise.
    *   **Digital Infrastructure:** Expanding internet penetration, smartphone usage, UPI.
    *   **Startup Ecosystem:** Thriving tech startup scene attracting significant investment.
    *   **Data Availability:** Vast population generating diverse datasets.

**III. Major Application Areas**
    *   **Healthcare:** Diagnostics, drug discovery, remote patient monitoring, personalized medicine.
    *   **Agriculture:** Crop yield p

In [16]:
print(final_state['blog'])

## India's AI Ascent: Charting a Course for Global Leadership

India is rapidly emerging as a formidable force in the global Artificial Intelligence landscape. What was once a futuristic concept is now deeply embedded across various sectors, driven by a national vision to become a global hub for AI innovation and implementation, encapsulated by the mantra "AI for All." This incredible momentum is fueled by a perfect storm of a vast data pool, a burgeoning skilled workforce, and robust government support.

The rise of AI in India isn't accidental; it's a meticulously nurtured phenomenon. **Government initiatives** like the "National Strategy for AI" and NITI Aayog's focused approach, coupled with the "Digital India" push, have laid a strong foundation. This framework is complemented by India's immense **talent pool** of STEM graduates and growing AI/ML expertise. The expanding **digital infrastructure**—with widespread internet penetration, smartphone usage, and the revolutionary UPI pa

In [17]:
print(final_state['evaluation'])

9.5/10
